# シーケンスレベル異常検知：窓分割なし前処理と検証

このノートブックでは、窓分割を行わずにシーケンス全体から抽出した特徴量を使用して異常検知を行います。

## 目的
1. 窓分割なしの前処理パイプラインを実行
2. シーケンス全体の特徴量を使った異常検知
3. `sequence_type='Target'`を異常として検出
4. 正常データでオートエンコーダーを学習
5. 異常検知性能の評価と可視化
6. 前処理の妥当性検証

## データフロー
```
生データ (train.csv)
↓
窓分割なし前処理 (NoWindowPreprocessor)
↓
シーケンス単位の特徴量抽出
↓
正常データでオートエンコーダー学習
↓
異常スコア計算
↓
異常検知性能評価
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pickle
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 1. 窓分割なし前処理パイプラインの実行

In [ ]:
# 前処理パイプラインのインポート
import sys
sys.path.append('../..')

from scripts.run_preprocessing_no_windows import NoWindowPreprocessor, load_yaml
from src.utils.pipeline import augment_handedness_flip
import yaml

print("前処理パイプラインをインポートしました")

In [ ]:
# 設定ファイルの読み込み
config_path = Path("../../config/config_v2.yaml")
config = load_yaml(config_path)

print("=== 設定情報 ===")
print(f"データディレクトリ: {config.get('data_dir', 'data')}")
print(f"出力ディレクトリ: {config.get('output_dir', 'output/experiments')}")

# 前処理設定
pp_config = config.get('preprocessing', {})
print(f"\n前処理設定:")
print(f"  センサー列: {len(config.get('sensor_acc_cols', []) + len(config.get('sensor_rot_cols', [])) + len(config.get('sensor_thm_cols', []))}個")
print(f"  人口統計列: {len(config.get('demographics_cols', []))}個")
print(f"  FFT帯域: {pp_config.get('fft_bands', [])}")

In [ ]:
# データの読み込み
data_dir = Path(config.get('data_dir', 'data'))

print("=== データ読み込み ===")
try:
    train_df = pd.read_csv(data_dir / "train.csv")
    train_demo = pd.read_csv(data_dir / "train_demographics.csv")
    
    # 人口統計データをマージ
    train_df = train_df.merge(train_demo, on="subject", how="left")
    
    print(f"訓練データ形状: {train_df.shape}")
    print(f"列数: {len(train_df.columns)}")
    print(f"サブジェクト数: {train_df['subject'].nunique()}")
    print(f"シーケンス数: {train_df['sequence_id'].nunique()}")
    
    # sequence_typeの確認
    if 'sequence_type' in train_df.columns:
        print(f"\nsequence_type分布:")
        print(train_df['sequence_type'].value_counts())
        
        # 異常データの定義
        is_anomaly = (train_df['sequence_type'] == 'Target')
        print(f"\n異常データ統計:")
        print(f"  正常サンプル数: {np.sum(~is_anomaly)}")
        print(f"  異常サンプル数: {np.sum(is_anomaly)}")
        print(f"  異常率: {np.mean(is_anomaly):.2%}")
    else:
        print("警告: sequence_type列が見つかりません")
        
except FileNotFoundError as e:
    print(f"エラー: データファイルが見つかりません - {e}")
    print("データパスを確認してください")
    raise

In [ ]:
# 窓分割なし前処理の実行
print("=== 窓分割なし前処理開始 ===")

# NoWindowPreprocessorの初期化
pp = NoWindowPreprocessor(config)

# フィッティング
print("前処理パイプラインをフィッティング中...")
pp.fit(train_df, use_cache=True)
print("フィッティング完了")

# 変換
print("データ変換中...")
train_data = pp.transform(train_df, use_cache=True)
print("変換完了")

print(f"\n=== 前処理結果 ===")
print(f"シーケンス数: {len(train_data['sequences'])}")
print(f"人口統計データ形状: {train_data['demographics'].shape}")
print(f"表形式特徴量形状: {train_data['tabular'].shape}")
print(f"ToF特徴量形状: {train_data['tof_features'].shape}")
print(f"ラベル数: {len(train_data['labels'])}")

# シーケンス情報の確認
print(f"\nシーケンス情報:")
for i, info in enumerate(train_data['info'][:5]):  # 最初の5個のみ表示
    print(f"  {i+1}. Subject {info['subject']}, Sequence {info['sequence_id']}, Length {info['length']}")

## 2. 異常検知用データの準備

In [ ]:
# 異常ラベルの設定
print("=== 異常検知用データ準備 ===")

# sequence_typeに基づく異常ラベルの設定
if 'sequence_type' in train_df.columns:
    # シーケンス単位で異常ラベルを作成
    sequence_anomaly_labels = []
    
    for info in train_data['info']:
        subject = info['subject']
        sequence_id = info['sequence_id']
        
        # 該当シーケンスのsequence_typeを取得
        seq_data = train_df[
            (train_df['subject'] == subject) & 
            (train_df['sequence_id'] == sequence_id)
        ]
        
        # sequence_typeが'Target'なら異常
        is_anomaly_seq = (seq_data['sequence_type'].iloc[0] == 'Target')
        sequence_anomaly_labels.append(is_anomaly_seq)
    
    is_anomaly = np.array(sequence_anomaly_labels)
    
    print(f"異常検知ラベル設定完了:")
    print(f"  正常シーケンス数: {np.sum(~is_anomaly)}")
    print(f"  異常シーケンス数: {np.sum(is_anomaly)}")
    print(f"  異常率: {np.mean(is_anomaly):.2%}")
    
else:
    print("警告: sequence_type列が見つかりません")
    print("仮想的な異常ラベルを作成します")
    
    # 仮想的な異常ラベル（実際のデータでは適切に設定）
    n_sequences = len(train_data['sequences'])
    is_anomaly = np.random.choice([True, False], size=n_sequences, p=[0.1, 0.9])
    
    print(f"仮想異常ラベル作成:")
    print(f"  正常シーケンス数: {np.sum(~is_anomaly)}")
    print(f"  異常シーケンス数: {np.sum(is_anomaly)}")
    print(f"  異常率: {np.mean(is_anomaly):.2%}")

# 表形式特徴量を異常検知に使用
X_tabular = train_data['tabular']
print(f"\n表形式特徴量形状: {X_tabular.shape}")
print(f"特徴量数: {X_tabular.shape[1]}")
print(f"シーケンス数: {X_tabular.shape[0]}")

## 3. オートエンコーダーの構築と学習

In [ ]:
class SequenceAutoencoder(tf.keras.Model):
    def __init__(self, input_dim, latent_dim=64, dropout_rate=0.3):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.dropout_rate = dropout_rate
        
        # エンコーダー
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Dense(256, activation='relu', name='encoder_dense1'),
            tf.keras.layers.BatchNormalization(name='encoder_bn1'),
            tf.keras.layers.Dropout(dropout_rate, name='encoder_dropout1'),
            
            tf.keras.layers.Dense(128, activation='relu', name='encoder_dense2'),
            tf.keras.layers.BatchNormalization(name='encoder_bn2'),
            tf.keras.layers.Dropout(dropout_rate, name='encoder_dropout2'),
            
            tf.keras.layers.Dense(latent_dim, activation='relu', name='latent')
        ])
        
        # デコーダー
        self.decoder = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu', name='decoder_dense1'),
            tf.keras.layers.BatchNormalization(name='decoder_bn1'),
            tf.keras.layers.Dropout(dropout_rate, name='decoder_dropout1'),
            
            tf.keras.layers.Dense(256, activation='relu', name='decoder_dense2'),
            tf.keras.layers.BatchNormalization(name='decoder_bn2'),
            tf.keras.layers.Dropout(dropout_rate, name='decoder_dropout2'),
            
            tf.keras.layers.Dense(input_dim, activation='linear', name='output')
        ])
    
    def call(self, x, training=False):
        encoded = self.encoder(x, training=training)
        decoded = self.decoder(encoded, training=training)
        return decoded
    
    def encode(self, x):
        """エンコードのみ"""
        return self.encoder(x)
    
    def get_anomaly_score(self, x):
        """異常スコアを計算"""
        decoded = self(x, training=False)
        reconstruction_error = tf.reduce_mean(tf.square(x - decoded), axis=1)
        return reconstruction_error

print("オートエンコーダークラスを定義しました")

In [ ]:
# オートエンコーダーの構築
input_dim = X_tabular.shape[1]
latent_dim = min(64, input_dim // 4)  # 入力次元の1/4、最大64

autoencoder = SequenceAutoencoder(input_dim, latent_dim)
autoencoder.build((None, input_dim))

print(f"=== オートエンコーダー情報 ===")
print(f"入力次元: {input_dim}")
print(f"潜在次元: {latent_dim}")
print(f"圧縮率: {input_dim / latent_dim:.1f}:1")
print(f"総パラメータ数: {autoencoder.count_params():,}")

# 正常データのみで学習
X_normal = X_tabular[~is_anomaly]
X_anomaly = X_tabular[is_anomaly]

X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)

print(f"\n=== 学習データ準備 ===")
print(f"正常データ数: {len(X_normal)}")
print(f"学習データ: {X_train.shape}")
print(f"検証データ: {X_val.shape}")
print(f"異常データ: {X_anomaly.shape}")

# オートエンコーダー学習
autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("\n=== オートエンコーダー学習開始 ===")
history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=50,  # デモ用に短縮
    batch_size=32,
    verbose=1
)

print("学習完了！")

## 4. 異常検知性能の評価

In [ ]:
# 異常検知性能の評価
print("=== 異常検知性能評価 ===")

# 異常スコア計算
anomaly_scores_all = autoencoder.get_anomaly_score(X_tabular).numpy()
anomaly_scores_normal = autoencoder.get_anomaly_score(X_normal).numpy()
anomaly_scores_anomaly = autoencoder.get_anomaly_score(X_anomaly).numpy()

print(f"正常データの異常スコア - 平均: {np.mean(anomaly_scores_normal):.6f}")
print(f"異常データの異常スコア - 平均: {np.mean(anomaly_scores_anomaly):.6f}")

# 閾値設定と性能評価
def evaluate_anomaly_detection(anomaly_scores, is_anomaly, threshold_percentiles=[90, 95, 97, 99]):
    results = {}
    normal_scores = anomaly_scores[~is_anomaly]
    
    for percentile in threshold_percentiles:
        threshold = np.percentile(normal_scores, percentile)
        predicted_anomalies = anomaly_scores > threshold
        
        accuracy = accuracy_score(is_anomaly, predicted_anomalies)
        precision = precision_score(is_anomaly, predicted_anomalies, zero_division=0)
        recall = recall_score(is_anomaly, predicted_anomalies, zero_division=0)
        f1 = f1_score(is_anomaly, predicted_anomalies, zero_division=0)
        
        results[f'{percentile}th_percentile'] = {
            'threshold': threshold,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
    
    return results

# 異常検知評価
anomaly_results = evaluate_anomaly_detection(anomaly_scores_all, is_anomaly)

print(f"\n=== 異常検知結果 ===")
for percentile, result in anomaly_results.items():
    print(f"{percentile}:")
    print(f"  精度: {result['accuracy']:.4f}")
    print(f"  適合率: {result['precision']:.4f}")
    print(f"  再現率: {result['recall']:.4f}")
    print(f"  F1スコア: {result['f1_score']:.4f}")

# ROC AUC計算
roc_auc = roc_auc_score(is_anomaly, anomaly_scores_all)
print(f"\nROC AUC: {roc_auc:.4f}")

## 5. 結果の可視化

In [ ]:
# 可視化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 異常スコアの分布
axes[0, 0].hist(anomaly_scores_normal, bins=30, alpha=0.7, label='正常', color='blue', density=True)
axes[0, 0].hist(anomaly_scores_anomaly, bins=30, alpha=0.7, label='異常', color='red', density=True)
axes[0, 0].set_title('異常スコアの分布')
axes[0, 0].set_xlabel('異常スコア')
axes[0, 0].set_ylabel('密度')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. 学習曲線
epochs = range(1, len(history.history['loss']) + 1)
axes[0, 1].plot(epochs, history.history['loss'], 'b-', label='学習損失')
axes[0, 1].plot(epochs, history.history['val_loss'], 'r-', label='検証損失')
axes[0, 1].set_title('学習曲線')
axes[0, 1].set_xlabel('エポック')
axes[0, 1].set_ylabel('損失 (MSE)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. ROC曲線
fpr, tpr, _ = roc_curve(is_anomaly, anomaly_scores_all)
axes[1, 0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
axes[1, 0].set_xlabel('偽陽性率 (FPR)')
axes[1, 0].set_ylabel('真陽性率 (TPR)')
axes[1, 0].set_title('ROC曲線')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. 閾値別性能
percentiles = [90, 95, 97, 99]
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
colors = ['blue', 'green', 'red', 'orange']

for i, metric in enumerate(metrics):
    values = [anomaly_results[f'{p}th_percentile'][metric] for p in percentiles]
    axes[1, 1].plot(percentiles, values, 'o-', color=colors[i], label=metric.capitalize())

axes[1, 1].set_xlabel('閾値パーセンタイル')
axes[1, 1].set_ylabel('スコア')
axes[1, 1].set_title('閾値別性能')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 6. 前処理の妥当性検証

In [ ]:
# 前処理の妥当性検証
print("=== 前処理の妥当性検証 ===")

# 特徴量の基本統計
print(f"\n表形式特徴量の基本統計:")
print(f"  平均: {np.mean(X_tabular):.4f}")
print(f"  標準偏差: {np.std(X_tabular):.4f}")
print(f"  最小値: {np.min(X_tabular):.4f}")
print(f"  最大値: {np.max(X_tabular):.4f}")
print(f"  欠損値: {np.isnan(X_tabular).sum()}")
print(f"  無限値: {np.isinf(X_tabular).sum()}")

# 正常・異常データの特徴量比較
print(f"\n正常・異常データの特徴量比較:")
print(f"  正常データ平均: {np.mean(X_normal):.4f}")
print(f"  異常データ平均: {np.mean(X_anomaly):.4f}")
print(f"  差の絶対値: {np.abs(np.mean(X_normal) - np.mean(X_anomaly)):.4f}")

# 特徴量の分散
feature_variances = np.var(X_tabular, axis=0)
print(f"\n特徴量の分散:")
print(f"  平均分散: {np.mean(feature_variances):.4f}")
print(f"  最大分散: {np.max(feature_variances):.4f}")
print(f"  最小分散: {np.min(feature_variances):.4f}")

# 前処理の妥当性評価
print(f"\n=== 前処理妥当性評価 ===")

validity_score = 0
max_score = 5

# 1. 欠損値チェック
if np.isnan(X_tabular).sum() == 0:
    print("✓ 欠損値なし")
    validity_score += 1
else:
    print(f"✗ 欠損値あり: {np.isnan(X_tabular).sum()}個")

# 2. 無限値チェック
if np.isinf(X_tabular).sum() == 0:
    print("✓ 無限値なし")
    validity_score += 1
else:
    print(f"✗ 無限値あり: {np.isinf(X_tabular).sum()}個")

# 3. 正常・異常データの分離性
if np.abs(np.mean(X_normal) - np.mean(X_anomaly)) > 0.1:
    print("✓ 正常・異常データに差あり")
    validity_score += 1
else:
    print("✗ 正常・異常データの差が小さい")

# 4. 異常検知性能
if roc_auc > 0.7:
    print("✓ 異常検知性能良好 (ROC AUC > 0.7)")
    validity_score += 1
else:
    print(f"✗ 異常検知性能要改善 (ROC AUC = {roc_auc:.3f})")

# 5. 特徴量の多様性
if np.std(feature_variances) > 0.1:
    print("✓ 特徴量に多様性あり")
    validity_score += 1
else:
    print("✗ 特徴量の多様性が低い")

print(f"\n前処理妥当性スコア: {validity_score}/{max_score} ({validity_score/max_score*100:.1f}%)")

if validity_score >= 4:
    print("✅ 前処理は妥当です")
elif validity_score >= 3:
    print("⚠️ 前処理は部分的に妥当です")
else:
    print("❌ 前処理の改善が必要です")

## 7. 結果の保存とまとめ

In [ ]:
# 結果保存
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

print("=== 結果保存 ===")
results_summary = {
    'preprocessing_info': {
        'processing_type': 'sequence_level_no_windows',
        'n_sequences': len(train_data['sequences']),
        'n_features': X_tabular.shape[1],
        'n_normal': np.sum(~is_anomaly),
        'n_anomaly': np.sum(is_anomaly),
        'anomaly_rate': float(np.mean(is_anomaly))
    },
    'model_info': {
        'input_dim': input_dim,
        'latent_dim': latent_dim,
        'compression_ratio': input_dim / latent_dim
    },
    'performance': {
        'roc_auc': roc_auc,
        'best_f1': max([result['f1_score'] for result in anomaly_results.values()])
    },
    'anomaly_detection_results': anomaly_results,
    'preprocessing_validity': {
        'score': validity_score,
        'max_score': max_score,
        'percentage': validity_score/max_score*100
    }
}

with open(output_dir / "sequence_anomaly_detection_results.json", 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)

print(f"結果を保存しました: {output_dir / 'sequence_anomaly_detection_results.json'}")

# まとめ
print(f"\n" + "="*60)
print(f"        シーケンスレベル異常検知 結果まとめ")
print(f"" + "="*60)
print(f"📊 データ情報:")
print(f"   ・シーケンス数: {len(train_data['sequences'])}")
print(f"   ・特徴量数: {X_tabular.shape[1]}")
print(f"   ・正常シーケンス数: {np.sum(~is_anomaly)}")
print(f"   ・異常シーケンス数: {np.sum(is_anomaly)}")
print(f"   ・異常率: {np.mean(is_anomaly):.2%}")

print(f"\n🏗️ モデル情報:")
print(f"   ・入力次元: {input_dim}")
print(f"   ・潜在次元: {latent_dim}")
print(f"   ・圧縮率: {input_dim / latent_dim:.1f}:1")

print(f"\n🎯 異常検知性能:")
print(f"   ・ROC AUC: {roc_auc:.4f}")
best_result = max(anomaly_results.values(), key=lambda x: x['f1_score'])
print(f"   ・最良F1スコア: {best_result['f1_score']:.4f}")

print(f"\n✅ 前処理の妥当性:")
print(f"   ・妥当性スコア: {validity_score}/{max_score} ({validity_score/max_score*100:.1f}%)")

print(f"\n💡 推奨事項:")
if roc_auc > 0.7 and validity_score >= 4:
    print(f"   - このアプローチは有効です")
    print(f"   - 窓分割なし前処理は妥当です")
    print(f"   - 異常スコアをタブラー特徴量に追加することを推奨")
elif roc_auc > 0.7:
    print(f"   - 異常検知性能は良好ですが、前処理の改善を推奨")
else:
    print(f"   - 特徴量エンジニアリングの見直しを推奨")
    print(f"   - より高次の特徴量や異なるモダリティの組み合わせを検討")

print(f"\n" + "="*60)
print(f"異常検知分析が完了しました！")
print(f"" + "="*60)